# Discover constructs and build a codebook

This tutorial starts with documents and a theoretical framework, rather than a
predefined codebook. It discovers constructs, consolidates their names, lets you
review and edit the groups, and builds a codebook for the coding pipeline.

**Requires the updated discovery module supplied with the discovery refactor.**
If you replaced the module while a notebook was open, restart the kernel first.
Use the Python environment where `llm_tracker` and Jupyter are installed.

Discovery and semantic merging make LLM calls. Manual edits, filtering, formatting,
and codebook creation run locally. An optional reevaluation step makes another
LLM call. No API key is stored in this notebook.


## 1. Set paths and options

Run the demo first, or set `use_demo_data=False` and point `input_path` to your data.
Relative paths are resolved from the notebook's working directory.

- TXT: one document per file; set `text_column=None` and `id_column=None`.
- CSV/XLSX/XLS: set `text_column` for one document per row; `id_column` is optional.
- A directory is read nonrecursively. The column settings apply to every tabular file.

For tabular data, omitting `text_column` treats the entire table as one document.
Set `model_name` to an available model ID for your provider; leaving it `None`
uses the package's configured default. Set `OPENROUTER_API_KEY` in your environment
or specify a local `.env` file. Never commit that file or print your key.


In [ ]:
from pathlib import Path

use_demo_data = True
input_path = Path("data/discovery_demo.csv")
output_root = Path("discovery_outputs")

text_column = "text"
id_column = "doc_id"
framework = "Cognitive behavioral therapy"  # Any framework name is accepted.

provider = "openrouter"
model_name = None  # Or your chosen provider model ID.
env_file = None  # Or Path("/absolute/path/to/.env").
max_retries = 2

fuzzy_threshold = 0.92  # None disables fuzzy spelling matching.
seed = 0
run_reevaluation = False  # True makes an additional LLM call after filtering.
n_codebook_examples = 3


In [ ]:
import json
from copy import deepcopy

import pandas as pd
from IPython.display import display

import llm_tracker.discovery as discovery_module
from llm_tracker.config import AnalyzerConfig
from llm_tracker.discovery_v2 import (
    DISCOVERY_PROMPT,
    MERGE_PROMPT,
    REEVALUATE_PROMPT,
    LLMTrackerDiscoverer,
    apply_merge_edits,
    build_codebook,
    codebook_to_frame,
    collapse_variants,
    filter_merged_constructs,
    format_merged_constructs,
    frame_to_codebook,
    merge_constructs,
    reevaluate_merged_constructs,
    save_codebook,
)

print("Discovery module:", discovery_module.__file__)
print("Working directory:", Path.cwd())


The demo cell creates a CSV only when `use_demo_data=True`. Change the input path
before using your own data. An existing demo file is left untouched.


In [ ]:
if use_demo_data:
    text_column, id_column = "text", "doc_id"
    input_path.parent.mkdir(parents=True, exist_ok=True)
    if not input_path.exists():
        demo = pd.DataFrame([
            {"doc_id": "demo_1", "text": "If I make one mistake at work, everyone will think I am incompetent."},
            {"doc_id": "demo_2", "text": "I assume my friend is angry whenever she takes a while to reply."},
            {"doc_id": "demo_3", "text": "I felt anxious about the meeting, so I avoided opening the invitation."},
            {"doc_id": "demo_4", "text": "I wrote down the evidence for and against my fear, which helped me consider another explanation."},
            {"doc_id": "demo_5", "text": "I took one small step toward the task instead of putting it off again."},
            {"doc_id": "demo_6", "text": "The delivery contained three blue folders and a box of paper clips."},
        ])
        demo.to_csv(input_path, index=False)
    display(pd.read_csv(input_path))

if not input_path.exists():
    raise FileNotFoundError(f"Update input_path; no input exists at {input_path.resolve()}")

discovery_dir = output_root / "discovery"
merge_dir = output_root / "merged"
edits_dir = output_root / "edited"
filtered_dir = output_root / "filtered"
reevaluation_dir = output_root / "reevaluated"
codebook_dir = output_root / "codebooks"
output_root.mkdir(parents=True, exist_ok=True)


## 2. Configure the discoverer and inspect its prompt

The discovery prompt uses `{framework}` and `{document}`. Keep these placeholders
when customizing it. Literal JSON braces in a template must be doubled: `{{` and `}}`.
Printing a template does not call the API.


In [ ]:
config_kwargs = {
    "provider": provider,
    "max_retries": max_retries,
    "temperature": 0.0,
}
if model_name is not None:
    config_kwargs["model_name"] = model_name
if env_file is not None:
    config_kwargs["api_key"] = str(env_file)

config = AnalyzerConfig(**config_kwargs)
discoverer = LLMTrackerDiscoverer(config=config)
print("Model:", config.model_name)
print(discoverer.discovery_prompt)


In [ ]:
# Demonstrate changing the prompt, then restore the default before discovery.
discoverer.discovery_prompt = DISCOVERY_PROMPT + "\nUse concise construct names and definitions."
print(discoverer.discovery_prompt)

discoverer.discovery_prompt = DISCOVERY_PROMPT
assert discoverer.discovery_prompt == DISCOVERY_PROMPT


## 3. Discover constructs in the documents

This cell makes LLM calls: normally one per document, with retries as configured.
Rerunning it sends every input document again. Later steps use the dataframe in
memory and do not repeat discovery.

The result has one row per supporting quote. A detected construct with no quote
gets an empty-quote row. Successful documents with no detected constructs have no
rows, but still count toward the frequency denominator. Failed documents do not.
The tutorial stops on failures so you can review them before interpreting results.


In [ ]:
discoveries, discovery_metadata, discovery_errors = discoverer.discover(
    path=input_path,
    framework=framework,
    text_column=text_column,
    id_column=id_column,
    output_dir=discovery_dir,
    save_format="both",
)

print("Successfully processed documents:", discoveries.attrs["total_documents"])
print("Discovery rows:", len(discoveries))
print("Failures:", len(discovery_errors))
display(discoveries)

if discovery_errors:
    display(pd.DataFrame([error.model_dump() for error in discovery_errors]))
    raise RuntimeError("Review the saved discovery errors before continuing. Rerunning discovery processes every document again.")
if discoveries.empty:
    raise RuntimeError("No constructs were discovered. Review the inputs, framework, and prompt before merging.")


## 4. Inspect name normalization before semantic merging

This step is local. Case, separators, and extra spaces are normalized; constituent
names become lowercase. Conservative fuzzy matching also combines close spellings.
It compares spelling, not meaning: review the match record. Set `fuzzy_threshold=None`
to use only ordinary name normalization.

`count` in this intermediate table counts distinct `(filename, doc_id)` pairs.
`instance_count` counts nonempty quote occurrences before deduplication. The final
merge table calls the document count `document_count`.


In [ ]:
normalized = collapse_variants(discoveries, fuzzy_threshold=fuzzy_threshold)
display(normalized[["construct", "count", "instance_count", "variants"]])
display(pd.DataFrame(normalized.attrs.get("name_matches", [])))


## 5. Merge constructs by meaning

Merging makes an LLM call to group related constituents, write merged names and
definitions, and assign prototypicality scores. Inspect or customize this template
using `{framework}`, `{constructs}`, and `{existing_note}`; the example restores
the default before running.


In [ ]:
print(MERGE_PROMPT)
merge_prompt = MERGE_PROMPT + "\nPrefer concise, descriptive group names."
print(merge_prompt)
merge_prompt = MERGE_PROMPT


In [ ]:
merged, merge_table = merge_constructs(
    df=discoveries,
    config=config,
    framework=framework,
    merge_prompt=merge_prompt,
    fuzzy_threshold=fuzzy_threshold,
    seed=seed,
    batch=False,
    max_prompt_chars=60000,
    output_dir=merge_dir,
    save_format="both",
)

display(format_merged_constructs(merge_table, n_quotes=3, seed=seed))


Each row describes one `construct_found` within a `merged_construct`.

| Column | Meaning |
|---|---|
| `document_count` | Distinct documents containing this constituent |
| `instance_count` | Nonempty quote occurrences, before duplicate quote text is removed |
| `document_frequency_percent` | `100 × document_count / total_documents` |
| `total_documents` | All successful documents, including those with no discoveries |
| `prototypicality` | LLM score from 0 to 1 for how representative the constituent is |
| `status` | `merged`, `standalone`, or `unplaced` |

The formatted view adds `document_frequency`, for example `20.0% (N=10)`.
The denominator is not the number of positive documents. A standalone construct
was kept independent; an unplaced construct was omitted by the merging response
and rescued into its own group. Missing scores remain missing.

Use the **raw `merge_table`** for later functions. The formatted view joins quotes
into display strings. For large inputs, `batch=True` splits merging into sequential
calls; this is different from the provider's asynchronous Batch API.


## 6. Review and apply manual grouping edits

The dictionary below is also saved as `merged_constructs_groups_<timestamp>.json`.
You can rename a key, move a constituent between lists, or omit a constituent to
remove it. Each known constituent may appear only once. This function makes no
LLM calls. For direct Python pasting, the following values contain only strings
and lists, so the printed JSON also works as a Python dictionary.


In [ ]:
grouping = {
    name: group["construct_found"].tolist()
    for name, group in merge_table.groupby("merged_construct", sort=False)
}
print(json.dumps(grouping, indent=2, ensure_ascii=False))


In [ ]:
# Replace this assignment with your edited dictionary, or edit the copy below.
edited_grouping = deepcopy(grouping)

# Examples, using names that actually occur in the printed dictionary:
# edited_grouping["New group name"] = edited_grouping.pop("Old group name")
# edited_grouping["Group name"].remove("constituent to remove")

edited_merged, edited_table = apply_merge_edits(
    table=merge_table,
    grouping=edited_grouping,
    # definitions={"New group name": "Your revised definition."},
    output_dir=edits_dir,
    save_format="both",
)
display(format_merged_constructs(edited_table, n_quotes=3, seed=seed))


Counts and quotes stay with their constituents. The denominator stays fixed.
Existing definitions are preserved where a source is identifiable; a new mixed
group may need an explicit definition. Moved members lose their old prototypicality
score because it described their previous group. A pure group rename retains
scores. Definitions can be outdated after membership changes; reevaluation below
can update names and definitions, but does not rescore prototypicality.


## 7. Filter the merge results

This is also local. Minimum thresholds are inclusive, and all supplied criteria
must pass. Removal names are matched using case/separator normalization. Unknown
removal names raise an error.

If `min_prototypicality` is supplied, missing scores are excluded—including
standalone/unplaced constructs and moved members. Leave it `None` to retain them.
Filtering never reduces the successful-document denominator.


In [ ]:
remove_constructs = []       # Constituent names to remove.
remove_merged_constructs = []  # Whole merged groups to remove.
min_document_count = 1
min_instance_count = 1
min_prototypicality = None

filtered_merged, filtered_table = filter_merged_constructs(
    table=edited_table,
    remove_constructs=remove_constructs,
    remove_merged_constructs=remove_merged_constructs,
    min_document_count=min_document_count,
    min_instance_count=min_instance_count,
    min_prototypicality=min_prototypicality,
    output_dir=filtered_dir,
    save_format="both",
)
display(format_merged_constructs(filtered_table, n_quotes=3, seed=seed))
print("Constituents remaining:", len(filtered_table))
print("Removed members recorded:", len(filtered_table.attrs.get("removed_constituents", [])))

if filtered_table.empty:
    raise RuntimeError("All constituents were removed. Relax the filters or revise the grouping before building a codebook.")


## 8. Optionally reevaluate merged names and definitions

Set `run_reevaluation=True` in the options cell to enable this extra LLM call.
The model receives the previous groups, current groups, and removed constituents.
It may revise names and definitions only; it cannot move or restore constituents.
Quotes, counts, the denominator, and existing/missing scores stay unchanged.

The template is available as `REEVALUATE_PROMPT`. A custom template uses
`{framework}`, `{previous}`, `{removed}`, and `{current}` and must keep the response
schema. The full prompt is checked against `max_prompt_chars` before sending.


In [ ]:
# Print this to inspect the reevaluation instructions without an API call:
# print(REEVALUATE_PROMPT)

if run_reevaluation:
    final_merged, final_table = reevaluate_merged_constructs(
        table=filtered_table,
        config=config,
        framework=framework,
        reevaluation_prompt=REEVALUATE_PROMPT,
        max_prompt_chars=60000,
        output_dir=reevaluation_dir,
        save_format="both",
    )
else:
    final_merged, final_table = filtered_merged, filtered_table
    print("Reevaluation skipped; using the locally reviewed and filtered result.")

display(format_merged_constructs(final_table, n_quotes=3, seed=seed))


## 9. Build, review, and save a codebook

Codebook creation is local. Each merged construct becomes a codebook entry with
its definition, sampled example quotes, and an initially empty counterexample
list. Quotes are deduplicated and sampled from the pooled quotes for each group;
they are not balanced across constituents.

Review definitions and examples before coding. Keep list columns as real Python
lists when editing; display strings can split quotes containing semicolons.


In [ ]:
codebook = build_codebook(
    merged=final_merged,
    name="Discovery tutorial codebook",
    version=1,
    n_examples=n_codebook_examples,
    seed=seed,
)
codebook_frame = codebook_to_frame(codebook)
display(codebook_frame)


In [ ]:
# Make any manual edits to codebook_frame before running this cell.
# Example:
# codebook_frame.at[0, "definition"] = "A clearer definition."
# codebook_frame.at[0, "counter_examples"] = ["A quote that should not count."]

if codebook_frame["definition"].fillna("").str.strip().eq("").any():
    raise ValueError("Fill in missing codebook definitions before saving.")

reviewed_codebook = frame_to_codebook(
    codebook_frame,
    metadata=codebook["metadata"],
)
saved_codebook_path = save_codebook(
    reviewed_codebook,
    codebook_dir / "discovered_codebook.json",
)
print("Use this codebook path in the coding pipeline:", saved_codebook_path.resolve())


## 10. Locate outputs and reload without repeating discovery

Files from one save share a UTC timestamp; another save creates new files.
Discovery saves its quote table, API metadata, errors, and context. Each merge,
edit, filter, or reevaluation save writes the review table, detailed dictionary,
simple grouping dictionary, name-match records, and context.

The discovery module uses the output directories passed above. This tutorial does
not register discovery runs in the coding/comparison master metadata CSV.


In [ ]:
print("Output root:", output_root.resolve())
for directory in (discovery_dir, merge_dir, edits_dir, filtered_dir, reevaluation_dir, codebook_dir):
    if directory.exists():
        print(f"\n{directory.name}/")
        for file in sorted(directory.iterdir()):
            if file.is_file():
                print("  ", file.name)


Optional reload examples follow. They are disabled by default. Copy exact filenames
from one matching timestamp; do not combine a table and context from different saves.
JSON table files preserve quote lists. CSV list cells require parsing before edits.

Discovery context restores the successful-document denominator, including empty
discoveries. Merge context additionally restores original groups, removed members,
and document memberships. Without memberships, a multi-member group's distinct
document count is unknown rather than an incorrect sum of overlapping counts.


In [ ]:
reload_saved_examples = False

if reload_saved_examples:
    # Replace these with matching files from your discovery save.
    discovery_table_path = discovery_dir / "discoveries_TIMESTAMP.json"
    discovery_context_path = discovery_dir / "discovery_context_TIMESTAMP.json"
    reloaded_discoveries = pd.read_json(discovery_table_path, dtype={"doc_id": str})
    reloaded_discoveries.attrs.update(json.loads(discovery_context_path.read_text(encoding="utf-8")))
    print("Restored successful-document total:", reloaded_discoveries.attrs["total_documents"])

    # Likewise, use matching merge table/context files from a saved stage.
    merge_table_path = filtered_dir / "merged_constructs_TIMESTAMP.json"
    merge_context_path = filtered_dir / "merge_context_TIMESTAMP.json"
    reloaded_table = pd.read_json(merge_table_path)
    reloaded_table.attrs.update(json.loads(merge_context_path.read_text(encoding="utf-8")))
    display(format_merged_constructs(reloaded_table, n_quotes=3, seed=seed))

    # Continue with apply_merge_edits(), filter_merged_constructs(), or
    # reevaluate_merged_constructs(). No need to call discover() again.


Once the codebook is ready, use it to encode documents and compare LLM coding
with human coding in the [main tutorial](tutorial.ipynb).

Place this notebook in the repository's `tutorials/` directory beside
`tutorial.ipynb` so that the relative link works.
